# SYMFLUENCE Tutorial 04a — Logan River Workshop (Lumped, Cloud Data)

## Introduction

This workshop notebook demonstrates basin-scale hydrological modeling using SYMFLUENCE's lumped representation approach for the Logan River at Logan, Utah using cloud-based data sources. The workflow includes:

1. **Configuration** — Set up a lumped basin model for the Logan River
2. **Domain Definition** — Delineate the watershed using TauDEM
3. **Data Acquisition** — Fetch RDRS forcing data and USGS streamflow observations from cloud sources
4. **Model Execution** — Run a hydrological model: SUMMA
5. **Evaluation & Calibration** — Assess model performance and calibrate parameters

The **Logan River at Logan** is a snow-dominated mountain watershed in the Bear River Range of the Wasatch Mountains. USGS station 10109000 provides streamflow observations. The watershed covers approximately 550 km² with elevations ranging from ~1,400 m to over 2,900 m.

### Switching Models

To use a different model, change the `model=` parameter in the configuration cell below. Supported models include: `'NGEN'`, `'HBV'`, `'SACSMA'`, `'SUMMA'`, `'HYPE'`, `'FUSE'`.

**Launch this notebook from the CLI:**
```bash
symfluence example launch 4a
```

In [1]:
# Environment verification
import sys
import os
import warnings
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Suppress experimental module warnings for cleaner output
warnings.filterwarnings('ignore', message='.*is an EXPERIMENTAL module.*')
warnings.filterwarnings('ignore', message='.*import failed.*')

print(f"Python executable: {sys.executable}")

# Verify SYMFLUENCE is available
try:
    import symfluence
    print(f"SYMFLUENCE version: {symfluence.__version__}")
    print(f"SYMFLUENCE location: {Path(symfluence.__file__).parent}")
except ImportError:
    print("ERROR: SYMFLUENCE not found. Please activate the symfluence environment.")
    sys.exit(1)

In [2]:
# Fix working directory if running from .ipynb_checkpoints
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")

# If we're in .ipynb_checkpoints, move up to parent directory
if '.ipynb_checkpoints' in str(current_dir):
    correct_dir = current_dir.parent
    os.chdir(correct_dir)
    print(f"Changed to: {Path.cwd()}")
else:
    print("Working directory is correct")

# Verify we're in the workshop notebooks directory
expected_notebook = Path.cwd() / '04a_logan_river_workshop_asyncdds.ipynb'
if not expected_notebook.exists():
    print(f"WARNING: Expected notebook not found at {expected_notebook}")
else:
    print("Notebook location verified")

## Step 1 — Configuration

Create a configuration for the Logan River lumped basin model. Key settings:
- **Domain**: Lumped (single HRU) representation
- **Forcing**: RDRS (Regional Deterministic Reanalysis System) — hourly gridded data
- **Observations**: USGS streamflow from station 10109000
- **Period**: 4 years (2018-2021) with 1-year spinup

In [3]:
# Step 1 — Create basin-scale configuration
import symfluence as symfluence_pkg
from symfluence import SYMFLUENCE
from symfluence.core.config.models import SymfluenceConfig

# ── Workshop toggle ──────────────────────────────────────────────────────────
# Set to True on JupyterHub / 2i2c to redirect all heavy I/O to /tmp/.
# On a local machine or HPC with fast home storage, leave as False.
use_tmp_dir = True
# ─────────────────────────────────────────────────────────────────────────────

def resolve_symfluence_paths():
    """Resolve SYMFLUENCE code/data paths for repo based and JupyterHub Docker image based symfluence installs."""
    pkg_dir = Path(symfluence_pkg.__file__).resolve().parent
    repo_root = None

    for candidate in [pkg_dir, *pkg_dir.parents]:
        if (
            (candidate / "src" / "symfluence").exists()
            or (candidate / ".git").exists()
            or (candidate / "pyproject.toml").exists()
        ):
            repo_root = candidate.resolve()
            break

    if repo_root is not None:
        code_dir = repo_root
        data_dir = repo_root.parent / "SYMFLUENCE_data"
        mode = "symfluence repository detected"
    else:
        code_dir = Path(os.environ.get("SYMFLUENCE_CODE_DIR", str(Path.home() / "SYMFLUENCE")))
        data_dir = Path(os.environ.get("SYMFLUENCE_DATA_DIR", str(Path.home() / "SYMFLUENCE_data")))
        mode = "installed package / JupyterHub fallback"

    return code_dir, data_dir, mode

code_dir, data_dir, path_mode = resolve_symfluence_paths()
original_data_dir = data_dir

if use_tmp_dir:
    data_dir = Path('/tmp/SYMFLUENCE_data')
    data_dir.mkdir(parents=True, exist_ok=True)
    # Symlink installs (TauDEM, model binaries) from the original location
    installs_link = data_dir / 'installs'
    if not installs_link.exists():
        installs_source = original_data_dir / 'installs'
        if installs_source.exists():
            installs_link.symlink_to(installs_source.resolve())
            print(f"  Symlinked installs: {installs_link} -> {installs_source.resolve()}")
        else:
            print(f"  WARNING: installs not found at {installs_source}")
    print(f"use_tmp_dir=True  -->  data directory redirected to {data_dir}")

if not data_dir.exists():
    raise RuntimeError(
        f"SYMFLUENCE data directory does not exist: {data_dir}\n"
        "Set SYMFLUENCE_DATA_DIR or create the directory before continuing."
    )

installs_dir = data_dir / 'installs'
if not installs_dir.exists():
    print(
        f"WARNING: installs directory not found at {installs_dir}.\n"
        "If using the JupyterHub/Docker image, ensure your installs symlink is configured."
    )

print(f"Notebook directory: {current_dir}")
print(f"SYMFLUENCE path resolution mode: {path_mode}")
print(f"SYMFLUENCE data directory: {data_dir}")
print(f"SYMFLUENCE code directory: {code_dir}")
print(f"SYMFLUENCE installs directory: {installs_dir}")

# Build config kwargs
config_kwargs = dict(
    # Basic identification
    domain_name='Logan_River_at_Logan',
    experiment_id='workshop_run_summa_asyncdds',

    # Paths
    SYMFLUENCE_DATA_DIR=str(data_dir),
    SYMFLUENCE_CODE_DIR=str(code_dir),
    TAUDEM_DIR='default',

    # Model
    model='SUMMA',
    SETTINGS_SUMMA_GRU_COUNT=1,
    params_to_calibrate=(
        'k_soil,'  # hydraulic conductivity of soil (m s-1)
        'theta_sat,'  # porosity (-)
        'aquiferBaseflowExp,'  # baseflow exponent (-)
        'aquiferBaseflowRate,'  # baseflow rate when aquifer storage = aquiferScaleFactor (m s-1)
        'qSurfScale,'  # scaling factor in the surface runoff parameterization (-)
        'summerLAI,'  # maximum leaf area index at the peak of the growing season (m2 m-2)
        'frozenPrecipMultip,'  # frozen precipitation multiplier (-)
        'Fcapil,'  # capillary retention as a fraction of the total pore volume (-)
        'tempCritRain,'  # critical temperature where precipitation is rain (K)
        'heightCanopyTop,'  # height of top of the vegetation canopy above ground surface (m)
        'heightCanopyBottom,'  # height of bottom of the vegetation canopy above ground surface (m)
        'windReductionParam,'  # canopy wind reduction parameter (-)
        'vGn_n'  # van Genuchten "n" parameter (-)
    ),
    basin_params_to_calibrate=(
        'routingGammaScale,'  # scale parameter in Gamma distribution used for sub-grid routing (s)
        'routingGammaShape'  # shape parameter in Gamma distribution used for sub-grid routing (-)
    ),

    # Simulation period (4 years: 2018-2021)
    time_start='2018-01-01 01:00',
    time_end='2021-12-31 23:00',
    spinup_period='2018-01-01, 2018-12-31',
    calibration_period='2019-01-01, 2019-12-31',
    evaluation_period='2020-01-01, 2020-12-31',

    # Spatial domain (Logan River at Logan, UT — USGS 10109000)
    pour_point_coords='41.743098/-111.786432',
    bounding_box_coords='42.15/-111.90/41.70/-111.40',
    definition_method='lumped',
    discretization='GRUs',
    lumped_watershed_method='TauDEM',

    # Data sources
    data_access='cloud',
    forcing_dataset='RDRS',
    forcing_measurement_height=2,
    dem_source='copdem90',
    download_dem=True,

    # Streamflow observations
    station_id='10109000',
    streamflow_data_provider='USGS',
    download_usgs_data=True,

    # Calibration
    OPTIMIZATION_METHODS=['iteration'],
    optimization_target='streamflow',
    iterative_optimization_algorithm='ASYNC-DDS',  # Parallel version of DDS
    SKIP_WARM_START=True,  # Defaults to False (seeds with previous run)
    INITIAL_GUESS=None,  # Defaults to None; can seed with a dict of parameter values
    PARAMETER_BOUNDS={  # Set explicit parameter bounds
        "k_soil": {"min": 1.0e-6, "max": 1.0e-4, 
                "transform": "log"},  # Default: [1.0e-8, 1.0e-2]
        "theta_sat": {"min": 0.4, "max": 0.7},  # Default: [0.35, 0.65]
        "aquiferBaseflowExp": {"min": 1.0, "max": 5.0},  # Default: [1.0, 5.0]
        "aquiferBaseflowRate": {"min": 1.0e-9, "max": 1.0e-6, 
                                "transform": "log"},  # Default: [1.0e-9, 1.0e-5]
        "qSurfScale": {"min": 1.0, "max": 100.0,
                    "transform": "log"},  # Default: [1.0, 100.0]
        "summerLAI": {"min": 2.0, "max": 6.0},  # Default: [0.01, 10.0]
        "frozenPrecipMultip": {"min": 0.25, "max": 1.25},  # Default: [0.5, 5.0]
        "Fcapil": {"min": 0.05, "max": 0.15},  # Default: [0.005, 0.2]
        "tempCritRain": {"min": 272.15, "max": 275.15},  # Default: [272.16, 274.16]
        "heightCanopyTop": {"min": 5.0, "max": 25.0},  # Default: [0.05, 100.0]
        "heightCanopyBottom": {"min": 1.0, "max": 5.0},  # Default: [0.0, 5.0]
        "windReductionParam": {"min": 0.25, "max": 0.75},  # Default: [0.25, 1.0]
        "vGn_n": {"min": 1.1, "max": 1.6},  # Default: [1.2, 4.0]
        "routingGammaScale": {"min": 100.0, "max": 100000.0, 
                            "transform": "log"},  # Default: [1.0, 100000.0]
        "routingGammaShape": {"min": 2.0, "max": 4.0},  # Default: [2.0, 5.0]
    },
    ASYNC_DDS_BATCH_SIZE=4,  # Batches are executed in parallel by worker processes
    ASYNC_DDS_POOL_SIZE=16,  # Number of candidate solutions to retain in the DDS pool
    DDS_R=0.1,  # Step size in normalized parameter space; default is 0.2
    num_processes=5,  # Use 5 processes for ASYNC-DDS (1 broker + 4 workers)
    optimization_metric='KGE',
    calibration_timestep='hourly',
    iterations=50,
)
config = SymfluenceConfig.from_minimal(**config_kwargs)

# Save configuration
config_path = Path('./config_logan_river_summa_asyncdds.yaml')
config_dict = config.to_dict(flatten=True)
import yaml
with open(config_path, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)
print(f"  Config saved to: {config_path}")

# Initialize SYMFLUENCE
symfluence = SYMFLUENCE(config)
project_dir = symfluence.managers['project'].setup_project()
pour_point_path = symfluence.managers['project'].create_pour_point()

print(f"\nProject structure created at: {project_dir}")
print(f"Pour point shapefile: {pour_point_path}")
print("=" * 80)

## Step 2 — Domain Definition

Delineate the Logan River watershed using TauDEM and create a single lumped HRU.

### Step 2a — Geospatial Attribute Acquisition

Acquire elevation, land cover, and soil data from cloud sources.

In [4]:
# Step 2a — Acquire geospatial attributes from cloud
symfluence.managers['data'].acquire_attributes()
print("Attribute acquisition complete")

### Step 2b — Watershed Delineation

Delineate the watershed boundary from the pour point using TauDEM.

In [5]:
# Step 2b — Watershed delineation
watershed_path = symfluence.managers['domain'].define_domain()
print(f"Watershed delineation complete")

### Step 2c — Domain Discretization

Create a single lumped HRU for the watershed.

In [6]:
# Step 2c — Discretization (single lumped HRU)
hru_path = symfluence.managers['domain'].discretize_domain()
print("Domain discretization complete")

### Step 2d — Visualization

Visualize the delineated watershed and pour point.

In [7]:
# Step 2d — Basin visualization
# Load spatial data
basin_path = project_dir / 'shapefiles' / 'river_basins' / f"{config.domain.name}_riverBasins_lumped.shp"
hru_file = project_dir / 'shapefiles' / 'catchment' / 'lumped' / config.domain.experiment_id / f"{config.domain.name}_HRUs_GRUs.shp"                     

watershed_gdf = gpd.read_file(str(basin_path))
hru_gdf = gpd.read_file(str(hru_file))
pour_point_gdf = gpd.read_file(pour_point_path)

# Calculate area (UTM Zone 12N for Utah)
watershed_proj = watershed_gdf.to_crs('EPSG:32612')
area_km2 = watershed_proj.geometry.area.sum() / 1e6

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
watershed_gdf.boundary.plot(ax=ax, color='blue', linewidth=2, label='Watershed')
hru_gdf.plot(ax=ax, facecolor='lightblue', edgecolor='blue', alpha=0.3)
pour_point_gdf.plot(ax=ax, color='red', markersize=150, marker='*', label=f'Pour Point (USGS {config.evaluation.streamflow.station_id})')

ax.set_title(f"Logan River at Logan\nArea: {area_km2:.0f} km²", fontweight='bold', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

print(f"Watershed area: {area_km2:.0f} km²")
print(f"Number of HRUs: {len(hru_gdf)} (lumped)")

## Step 3 — Data Acquisition and Preprocessing

Fetch forcing data (RDRS) and streamflow observations (USGS) from cloud sources.

### Step 3a — USGS Streamflow Observations

Download and process USGS streamflow data for station 10109000.

In [8]:
# Step 3a — Download and process USGS streamflow data
symfluence.managers['data'].process_observed_data()                                                                                                      
print("USGS streamflow data acquisition complete")

### Step 3b — RDRS Meteorological Forcing

Download RDRS forcing data from cloud. RDRS (Regional Deterministic Reanalysis System) provides:
- Hourly temporal resolution
- Complete forcing variables: precipitation, temperature, humidity, wind, radiation, pressure

In [9]:
# Step 3b — Acquire RDRS forcing data from cloud
symfluence.managers['data'].acquire_forcings()
print("RDRS forcing acquisition complete")

### Step 3c — Model-Agnostic Preprocessing

Standardize forcing data: variable names, units, and spatial averaging over the watershed.

In [10]:
# Step 3c — Model-agnostic preprocessing
symfluence.managers['data'].run_model_agnostic_preprocessing()
print("Model-agnostic preprocessing complete")

## Step 4 — Model Configuration and Execution

Run model-specific preprocessing and execute the selected model.

In [11]:
# Step 4a — Model-specific preprocessing
print(f"Preprocessing for {config.model.hydrological_model}...")
symfluence.managers['model'].preprocess_models()
print(f"{config.model.hydrological_model} preprocessing complete")

In [12]:
# Step 4b — Model execution
print(f"Running {config.model.hydrological_model}...")
print(f"Simulation period: {config.domain.time_start} to {config.domain.time_end}")
symfluence.managers['model'].run_models()
print("Model execution complete")

# Step 4c — Post-processing (extract streamflow to standardized results CSV)
print(f"Post-processing for {config.model.hydrological_model}")
symfluence.managers['model'].postprocess_results()
print("Post-processing complete")

## Step 5 — Streamflow Evaluation

Now that the model has produced a simulated hydrograph, we compare it to the USGS observations using three standard metrics: Nash–Sutcliffe Efficiency (NSE), Kling–Gupta Efficiency (KGE), and percent bias (PBIAS). Three small data-handling steps happen before the metrics are computed: the simulated discharge is resampled to daily means to match the observation frequency; the spinup period specified in the configuration is dropped because the model's internal storages haven't equilibrated yet and including those values would penalize the metrics unfairly; and the simulated and observed series are aligned on their common dates with NaN pairs removed. The metrics reported here are for the uncalibrated model — default parameters straight from the config. Step 5b will then run calibration and we can see how much these numbers improve.

In [13]:
model_name = config.model.hydrological_model
experiment_id = config.domain.experiment_id

# Load results CSV (written by postprocessor)
results_file = project_dir / "results" / f"{experiment_id}_results.csv"
if not results_file.exists():
    raise FileNotFoundError(f"No results file found at: {results_file}")

results_df = pd.read_csv(results_file, index_col=0, parse_dates=True)
print(f"Loaded results: {results_file}")
print(f"Columns: {list(results_df.columns)}")

# Find simulation column (any *_discharge_cms column)
sim_cols = [c for c in results_df.columns if 'discharge' in c.lower() and 'obs' not in c.lower()]
if not sim_cols:
    raise ValueError(f"No discharge column found. Available: {list(results_df.columns)}")
sim_col = sim_cols[0]
print(f"Using simulation column: {sim_col}")

# Load observed streamflow
obs_dir = project_dir / "data" / "observations" / "streamflow" / "preprocessed"
obs_path = obs_dir / f"{config.domain.name}_streamflow_processed.csv"
if obs_path.exists():
    obs_df = pd.read_csv(obs_path)
    obs_df['datetime'] = pd.to_datetime(obs_df['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    obs_df.set_index('datetime', inplace=True)
    obs_daily = obs_df['discharge_cms'].resample('D').mean()
else:
    print(f"Warning: observations not found at {obs_path}")
    obs_daily = None

# Align simulation and observations
sim_series = results_df[sim_col].resample('D').mean()

# Define post-spinup period
spinup_end = pd.to_datetime(config.domain.spinup_period.split(',')[1].strip())
sim_series = sim_series[sim_series.index > spinup_end]

common_idx = sim_series.index.intersection(obs_daily.index)
obs_valid = obs_daily.loc[common_idx].dropna()
sim_valid = sim_series.loc[obs_valid.index]

# Remove NaN pairs
mask = ~(np.isnan(obs_valid.values) | np.isnan(sim_valid.values))
obs_clean = obs_valid[mask]
sim_clean = sim_valid[mask]

print(f"\nExcluding spinup up to: {spinup_end}")
print(f"Post-spinup period: {obs_clean.index[0]} to {obs_clean.index[-1]}")
print(f"Valid data points: {len(obs_clean)}")

# Metrics
def nse(obs, sim):
    return float(1 - np.sum((obs - sim)**2) / np.sum((obs - obs.mean())**2))

def kge(obs, sim):
    r = obs.corr(sim)
    alpha = sim.std() / obs.std()
    beta = sim.mean() / obs.mean()
    return float(1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2))

def pbias(obs, sim):
    return float(100 * (sim.sum() - obs.sum()) / obs.sum())

metrics = {
    'NSE': round(nse(obs_clean, sim_clean), 3),
    'KGE': round(kge(obs_clean, sim_clean), 3),
    'PBIAS': round(pbias(obs_clean, sim_clean), 1)
}

print(f"\nPerformance Metrics ({model_name}, Uncalibrated):")
for k, v in metrics.items():
    print(f"  {k}: {v}")


In [14]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series (top left)
axes[0, 0].plot(obs_clean.index, obs_clean.values, 'b-', label='Observed (USGS)', linewidth=1.2, alpha=0.7)
axes[0, 0].plot(sim_clean.index, sim_clean.values, 'r-', label=f'Simulated ({model_name})', linewidth=1.2, alpha=0.7)
axes[0, 0].set_ylabel('Discharge (m\u00b3/s)')
axes[0, 0].set_title('Streamflow Time Series')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].text(0.02, 0.95, f"NSE: {metrics['NSE']}\nKGE: {metrics['KGE']}\nBias: {metrics['PBIAS']}%",
                transform=axes[0, 0].transAxes, verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.8), fontsize=9)

# Scatter (top right)
axes[0, 1].scatter(obs_clean, sim_clean, alpha=0.5, s=10)
max_val = max(obs_clean.max(), sim_clean.max())
axes[0, 1].plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
axes[0, 1].set_xlabel('Observed (m\u00b3/s)')
axes[0, 1].set_ylabel('Simulated (m\u00b3/s)')
axes[0, 1].set_title('Observed vs Simulated')
axes[0, 1].grid(True, alpha=0.3)

# Monthly climatology (bottom left)
monthly_obs = obs_clean.groupby(obs_clean.index.month).mean()
monthly_sim = sim_clean.groupby(sim_clean.index.month).mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1, 0].plot(monthly_obs.index, monthly_obs.values, 'b-o', label='Observed', markersize=6)
axes[1, 0].plot(monthly_sim.index, monthly_sim.values, 'r-o', label='Simulated', markersize=6)
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(month_names)
axes[1, 0].set_ylabel('Mean Discharge (m\u00b3/s)')
axes[1, 0].set_title('Seasonal Flow Regime')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Flow duration curve (bottom right)
obs_sorted = obs_clean.sort_values(ascending=False)
sim_sorted = sim_clean.sort_values(ascending=False)
obs_ranks = np.arange(1., len(obs_sorted) + 1) / len(obs_sorted) * 100
sim_ranks = np.arange(1., len(sim_sorted) + 1) / len(sim_sorted) * 100
axes[1, 1].semilogy(obs_ranks, obs_sorted, 'b-', label='Observed', linewidth=2)
axes[1, 1].semilogy(sim_ranks, sim_sorted, 'r-', label='Simulated', linewidth=2)
axes[1, 1].set_xlabel('Exceedance Probability (%)')
axes[1, 1].set_ylabel('Discharge (m\u00b3/s)')
axes[1, 1].set_title('Flow Duration Curve')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Logan River at Logan \u2014 Lumped {model_name} Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5b — Model Calibration

Calibrate model parameters using the DDS (Dynamically Dimensioned Search) algorithm to improve model performance. The calibration optimizes KGE over the calibration period.

In [15]:
# Step 5b — Run calibration
print(f"Starting calibration...")
print(f"Algorithm: {config.optimization.algorithm}")
print(f"Metric: {config.optimization.metric}")
print(f"Iterations: {config.optimization.iterations}")
print(f"Calibration period: {config.domain.calibration_period}")

results_file = symfluence.managers['optimization'].calibrate_model()
print(f"\nCalibration complete!")
print(f"Results file: {results_file}")

### View Calibration Results

In [16]:
# Load and display calibration results
import json as _json

cal_results = pd.read_csv(results_file)
cal_results['best_score'] = cal_results['score'].cummax()

print("Calibration Progress:")
print(f"  Best {config.optimization.metric}: {cal_results['best_score'].iloc[-1]:.4f}")
print(f"  Initial {config.optimization.metric}: {cal_results['best_score'].iloc[0]:.4f}")
print(f"  Improvement: {cal_results['best_score'].iloc[-1] - cal_results['best_score'].iloc[0]:.4f}")

# --- Load final evaluation metrics and calibrated simulation ---
opt_dir = Path(results_file).parent
fe_dir = opt_dir / 'final_evaluation'

# Final evaluation JSON (calibration + evaluation period metrics)
fe_metrics = {}
fe_jsons = sorted(opt_dir.glob('*_final_evaluation.json'))
if fe_jsons:
    with open(fe_jsons[0]) as _f:
        fe_data = _json.load(_f)
    fe_metrics['calib'] = {k.replace('Calib_', ''): v for k, v in fe_data.get('calibration_metrics', {}).items()}
    fe_metrics['eval'] = {k.replace('Eval_', ''): v for k, v in fe_data.get('evaluation_metrics', {}).items()}
    print(f"\nCalibration period:  KGE={fe_metrics['calib'].get('KGE', 'N/A'):.3f}  "
          f"NSE={fe_metrics['calib'].get('NSE', 'N/A'):.3f}")
    print(f"Evaluation period:   KGE={fe_metrics['eval'].get('KGE', 'N/A'):.3f}  "
          f"NSE={fe_metrics['eval'].get('NSE', 'N/A'):.3f}")

# Use the same post-spinup output window for final comparison
output_start = pd.to_datetime('2019-01-01')
output_end = pd.to_datetime('2022-01-01')

# Load calibrated streamflow from final_evaluation outputs
cal_sim = None
if fe_dir.exists():
    # Prefer CSV files when final evaluation already saved discharge
    for csv_path in sorted(fe_dir.glob('*.csv')):
        _df = pd.read_csv(csv_path, parse_dates=['datetime']).set_index('datetime')
        for col_name in [
            'streamflow_cms',
            'discharge_cms',
            f'{model_name}_discharge_cms',
        ]:
            if col_name in _df.columns:
                cal_sim = _df[col_name]
                break
        if cal_sim is not None:
            break
    if cal_sim is None:
        # Use shared streamflow evaluator for NetCDF model outputs
        from symfluence.evaluation.evaluators.streamflow import StreamflowEvaluator

        streamflow_evaluator = StreamflowEvaluator(config, project_dir=project_dir)
        sim_files = streamflow_evaluator.get_simulation_files(fe_dir)
        if sim_files:
            cal_sim = streamflow_evaluator.extract_simulated_data(sim_files)

if cal_sim is not None:
    cal_sim = cal_sim[
        (cal_sim.index >= output_start) & (cal_sim.index < output_end)
    ]

# Load observed streamflow
obs_dir = project_dir / "data" / "observations" / "streamflow" / "preprocessed"
obs_path = obs_dir / f"{config.domain.name}_streamflow_processed.csv"
cal_obs = None
if obs_path.exists():
    _obs = pd.read_csv(obs_path)
    _obs['datetime'] = pd.to_datetime(_obs['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    _obs.set_index('datetime', inplace=True)
    cal_obs = _obs['discharge_cms'].resample('D').mean()

# --- Build figure ---
if cal_sim is not None and cal_obs is not None:
    cal_sim_daily = cal_sim.resample('D').mean()
    common = cal_sim_daily.index.intersection(cal_obs.index)
    obs_c = cal_obs.loc[common].dropna()
    sim_c = cal_sim_daily.loc[obs_c.index]
    valid = ~(np.isnan(obs_c.values) | np.isnan(sim_c.values))
    obs_c, sim_c = obs_c[valid], sim_c[valid]

    cal_nse = round(nse(obs_c, sim_c), 3)
    cal_kge = round(kge(obs_c, sim_c), 3)
    cal_pbias = round(pbias(obs_c, sim_c), 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # (0,0) Calibration trajectory
    axes[0, 0].plot(cal_results['iteration'], cal_results['best_score'], 'b-', linewidth=2)
    axes[0, 0].scatter(cal_results['iteration'].iloc[-1], cal_results['best_score'].iloc[-1],
                       color='red', s=80, zorder=5, label=f"Final: {cal_results['best_score'].iloc[-1]:.3f}")
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel(f'Best {config.optimization.metric}')
    axes[0, 0].set_title(f'Calibration Trajectory ({config.optimization.algorithm})')
    axes[0, 0].legend(loc='lower right')
    axes[0, 0].grid(True, alpha=0.3)

    # (0,1) Time series — calibrated
    axes[0, 1].plot(obs_c.index, obs_c.values, 'b-', label='Observed (USGS)', linewidth=1.2, alpha=0.7)
    axes[0, 1].plot(sim_c.index, sim_c.values, 'r-', label=f'Calibrated ({model_name})', linewidth=1.2, alpha=0.7)
    axes[0, 1].set_ylabel('Discharge (m³/s)')
    axes[0, 1].set_title('Calibrated Streamflow')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].text(0.02, 0.95, f"NSE: {cal_nse}\nKGE: {cal_kge}\nBias: {cal_pbias}%",
                    transform=axes[0, 1].transAxes, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.8), fontsize=9)

    # (1,0) Seasonal flow regime
    mo = obs_c.groupby(obs_c.index.month).mean()
    ms = sim_c.groupby(sim_c.index.month).mean()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[1, 0].plot(mo.index, mo.values, 'b-o', label='Observed', markersize=6)
    axes[1, 0].plot(ms.index, ms.values, 'r-o', label='Calibrated', markersize=6)
    axes[1, 0].set_xticks(range(1, 13))
    axes[1, 0].set_xticklabels(month_names)
    axes[1, 0].set_ylabel('Mean Discharge (m³/s)')
    axes[1, 0].set_title('Seasonal Flow Regime')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # (1,1) Flow duration curve
    obs_s = obs_c.sort_values(ascending=False)
    sim_s = sim_c.sort_values(ascending=False)
    obs_r = np.arange(1., len(obs_s) + 1) / len(obs_s) * 100
    sim_r = np.arange(1., len(sim_s) + 1) / len(sim_s) * 100
    axes[1, 1].semilogy(obs_r, obs_s, 'b-', label='Observed', linewidth=2)
    axes[1, 1].semilogy(sim_r, sim_s, 'r-', label='Calibrated', linewidth=2)
    axes[1, 1].set_xlabel('Exceedance Probability (%)')
    axes[1, 1].set_ylabel('Discharge (m³/s)')
    axes[1, 1].set_title('Flow Duration Curve')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f'Logan River at Logan — Calibrated {model_name} Results',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cal_results['iteration'], cal_results['best_score'], 'b-', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(f'Best {config.optimization.metric}')
    ax.set_title('Calibration Progress')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    if cal_sim is None:
        print("No calibrated simulation output found in final_evaluation/")
    if cal_obs is None:
        print("No observed streamflow found for comparison")

## Step 6 — Benchmarking against reference predictors
A KGE of 0.80 sounds good, but good compared to what? For Logan River — a snow-dominated catchment with a strong annual cycle — streamflow can be predicted surprisingly well without any physics at all. Here we compute three reference predictors trained on the calibration period and evaluated on the same evaluation period as the model:

Mean flow — a constant prediction equal to the calibration-period mean. The lower bound: how much does the seasonal cycle alone explain?
Monthly climatology — the mean flow for each calendar month, repeated across years.
Daily climatology — the mean flow for each day-of-year. The strongest pure-climatology benchmark.

If the calibrated model doesn't beat daily climatology, it means the model is reproducing the seasonal cycle but not adding much beyond it. If it does beat the climatology — by how much? Knoben (2024) argues this comparison should be standard practice in every hydrological model evaluation.

In [17]:
# Step 6 — Benchmarking
opt_dir = Path(results_file).parent
fe_dir = opt_dir / 'final_evaluation'
 
# Load calibrated simulation
cal_sim_daily = None
if 'cal_sim' in globals() and cal_sim is not None:
    cal_sim_daily = cal_sim.resample('D').mean()
else:
    for csv_path in sorted(fe_dir.glob('*.csv')):
        _df = pd.read_csv(csv_path, parse_dates=['datetime']).set_index('datetime')
        for col in ['streamflow_cms', 'discharge_cms', f'{model_name}_discharge_cms']:
            if col in _df.columns:
                cal_sim_daily = _df[col].resample('D').mean()
                break
        if cal_sim_daily is not None:
            break
    if cal_sim_daily is None and fe_dir.exists():
        from symfluence.evaluation.evaluators.streamflow import StreamflowEvaluator

        streamflow_evaluator = StreamflowEvaluator(config, project_dir=project_dir)
        sim_files = streamflow_evaluator.get_simulation_files(fe_dir)
        if sim_files:
            cal_sim_daily = streamflow_evaluator.extract_simulated_data(sim_files).resample('D').mean()
if cal_sim_daily is None:
    raise FileNotFoundError(f"No calibrated streamflow output found in {fe_dir}")
 
# Load observations
obs_path = project_dir / "data" / "observations" / "streamflow" / "preprocessed" / f"{config.domain.name}_streamflow_processed.csv"
_obs = pd.read_csv(obs_path)
_obs['datetime'] = pd.to_datetime(_obs['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
_obs.set_index('datetime', inplace=True)
obs_daily = _obs['discharge_cms'].resample('D').mean()
 
# Periods
cal_start, cal_end = [pd.to_datetime(d.strip()) for d in config.domain.calibration_period.split(',')]
eval_start, eval_end = [pd.to_datetime(d.strip()) for d in config.domain.evaluation_period.split(',')]
obs_cal = obs_daily.loc[cal_start:cal_end].dropna()
obs_eval = obs_daily.loc[eval_start:eval_end].dropna()
 
sim_eval = cal_sim_daily.loc[obs_eval.index].dropna()
obs_eval_aligned = obs_eval.loc[sim_eval.index]
 
# Benchmarks (trained on calibration period, applied to evaluation period)
mean_bench = pd.Series(obs_cal.mean(), index=obs_eval_aligned.index)
 
monthly_clim = obs_cal.groupby(obs_cal.index.month).mean()
monthly_bench = pd.Series([monthly_clim[m] for m in obs_eval_aligned.index.month],
                          index=obs_eval_aligned.index)
 
doy_clim = obs_cal.groupby(obs_cal.index.dayofyear).mean()
doy_full = pd.Series(index=range(1, 367), dtype=float)
doy_full.update(doy_clim)
doy_full = doy_full.interpolate().ffill().bfill()
daily_bench = pd.Series([doy_full[d] for d in obs_eval_aligned.index.dayofyear],
                        index=obs_eval_aligned.index)
 
# NaN-safe KGE (constant predictions have zero variance — Pearson r is undefined)
def kge_safe(obs, sim):
    if float(sim.std()) == 0.0 or float(obs.std()) == 0.0:
        return float('nan')
    return kge(obs, sim)
 
benchmark_kge = {
    'Mean flow': kge_safe(obs_eval_aligned, mean_bench),
    'Monthly climatology': kge_safe(obs_eval_aligned, monthly_bench),
    'Daily climatology': kge_safe(obs_eval_aligned, daily_bench),
    f'Calibrated {model_name}': kge_safe(obs_eval_aligned, sim_eval),
}
 
print("Evaluation-period KGE (higher is better; max = 1.0):")
for name, score in benchmark_kge.items():
    if np.isnan(score):
        print(f"  {name:<30} n/a (constant predictor — variance = 0)")
    else:
        print(f"  {name:<30} {score:6.3f}")
 
# Plot only finite-KGE entries
plot_items = [(k, v) for k, v in benchmark_kge.items() if not np.isnan(v)]
names = [k for k, _ in plot_items]
scores = [v for _, v in plot_items]
bar_colors = ['#2E86AB' if model_name in n else '#888888' for n in names]
 
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names, scores, color=bar_colors, edgecolor='black')
 
benchmark_scores = [v for n, v in plot_items if model_name not in n]
if benchmark_scores:
    best_bench = max(benchmark_scores)
    ax.axvline(x=best_bench, color='red', linestyle='--', alpha=0.6,
               label=f'Best benchmark = {best_bench:.2f}')
    ax.legend(loc='lower right')
 
ax.set_xlabel('KGE (evaluation period)')
ax.set_title(f'Logan River — does {model_name} beat what we could predict without it?')
ax.grid(True, axis='x', alpha=0.3)
ax.set_xlim(min(min(scores) - 0.1, -0.2), 1.05)
plt.tight_layout()
plt.show()
 
